# RQFormer-2 Kaggle Full Workflow

Notebook này chạy trực tiếp toàn bộ workflow trên Kaggle, **không phụ thuộc `tools/run_all_rqformer_visualize.sh`**.

Pipeline:

1. Clone source từ GitHub hoặc fallback từ Kaggle input.
2. Link dataset đúng cấu trúc config gốc.
3. Patch config `_base_` từ `mmrotate::...` sang local base config để tránh lỗi `.mim/model-index.yml`.
4. Setup môi trường OpenMMLab/MMRotate.
5. Train hoặc resume từng dataset.
6. Test và lưu `predictions.pkl`.
7. Sinh chart/confusion nếu script/log có đủ.
8. Sinh bbox visualization và heatmap attention mẫu.
9. Ghi `summary_results.md`.
10. Zip output để tải về hoặc dùng resume.


## 0. Fine-tuning có phù hợp không?

Có. Source hiện tại vẫn dùng pipeline train/test gốc (`tools/train.py`, `tools/test.py`). Các module cải tiến nằm trong decoder/RRoI attention, có tham số học được và đi qua optimizer như layer bình thường.

Lưu ý:

- Không có checkpoint `.pth` vẫn train được từ đầu, nhưng lâu hơn.
- Nếu có checkpoint gốc, fine-tune nhanh hơn; các layer mới có thể báo `missing keys`, đây là bình thường.
- Chạy từng dataset nếu GPU time ít: `DATASETS="dior"`, `"dotav1_0"`, hoặc `"dotav1_5"`.
- Không visualize toàn bộ dataset; mặc định chỉ xuất `SAMPLE_IMAGES=10`.


## 1. Cấu hình Kaggle

Chỉnh cell này rồi bấm **Run All**.


In [ ]:
from pathlib import Path

# ===== Git source =====
REPO_GIT_URL = "https://github.com/hungnp-dev/RQFormer.git"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"  # nếu repo private
REPO_DIR = Path("/kaggle/working/RQFormer")

# Fallback nếu clone GitHub lỗi: upload source repo thành Kaggle Dataset tại path này.
REPO_INPUT_FALLBACK = Path("/kaggle/input/rqformer-source/RQFormer")

# ===== Dataset path =====
DATA_ROOT = Path("/kaggle/input/datasets/jks010/rqformer-dataset/data")
DATA_INPUTS = {
    "dior": DATA_ROOT / "DIOR",
    "dotav1_0": DATA_ROOT / "split_ss_dota",
    "dotav1_5": DATA_ROOT / "split_ss_dota1_5",
}

# ===== Optional checkpoint path =====
CKPT_INPUT = None  # Không dùng checkpoint ngoài; train từ đầu hoặc resume từ work_dirs
CKPT_DIR = Path("/kaggle/working/pth")

# ===== Output =====
WORK_ROOT = Path("/kaggle/working/work_dirs")

# ===== Dataset selection =====
DATASETS = "all"  # "all", "dior", "dotav1_0", "dotav1_5", "dior,dotav1_0"

# ===== Runtime controls =====
DEVICE = "0"
BATCH_SIZE = 2  # bám config gốc DIOR/DOTA; giảm xuống 1 nếu GPU thiếu VRAM
NUM_WORKERS = 2
SAMPLE_IMAGES = 10
SCORE_THR = 0.3
FORCE = 0
MAX_EPOCHS = None  # ví dụ 1 hoặc 3 để test nhanh; None = giữ config gốc
LR = None          # ví dụ 1e-5; None = giữ config gốc

# ===== Workflow stages =====
RUN_SETUP = 1
RUN_TRAIN = 1
RUN_TEST = 1
RUN_CHARTS = 1
RUN_CONFUSION = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
RUN_SUMMARY = 1

print("REPO_GIT_URL =", REPO_GIT_URL)
print("DATA_ROOT =", DATA_ROOT)
print("DATASETS =", DATASETS)
print("WORK_ROOT =", WORK_ROOT)


## 2. Khai báo 3 job dataset/config


In [ ]:
JOBS = {
    "dior": {
        "title": "RQFormer | DIOR-R | R50 | 3x | Query 500 | t0.85",
        "dataset_label": "DIOR-R",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.85_3x_dior.pth",
        "data_link": Path("data/DIOR"),
        "data_input": DATA_INPUTS["dior"],
        "image_source": Path("data/DIOR/JPEGImages-test"),
    },
    "dotav1_0": {
        "title": "RQFormer | DOTA-v1.0 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.0",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.0.pth",
        "data_link": Path("data/split_ss_dota"),
        "data_input": DATA_INPUTS["dotav1_0"],
        "image_source": Path("data/split_ss_dota/trainval/images"),
    },
    "dotav1_5": {
        "title": "RQFormer | DOTA-v1.5 | R50 | 2x | Query 500 | t0.9",
        "dataset_label": "DOTA-v1.5",
        "config": "projects/RQFormer/configs/rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.py",
        "ckpt": "rroiformer_le90_r50_q500_layer2_sq1_dq1_t0.9_2x_dotav1.5.pth",
        "data_link": Path("data/split_ss_dota1_5"),
        "data_input": DATA_INPUTS["dotav1_5"],
        "image_source": Path("data/split_ss_dota1_5/trainval/images"),
    },
}

def selected_names():
    if DATASETS == "all":
        return list(JOBS)
    names = [x.strip() for x in DATASETS.split(",") if x.strip()]
    unknown = [x for x in names if x not in JOBS]
    if unknown:
        raise ValueError(f"Unknown DATASETS: {unknown}. Valid: {list(JOBS)}")
    return names

print("Selected jobs:", selected_names())


## 3. Clone source, link data, patch config base

Cell này xử lý lỗi bạn gặp: `mmrotate/.mim/model-index.yml` bằng cách sửa 3 config RQFormer sang dùng base config local:

- `../../../configs/_base_/datasets/dior.py`
- `../../../configs/_base_/datasets/dota.py`
- `../../../configs/_base_/datasets/dotav15.py`
- `../../../configs/_base_/schedules/schedule_3x.py`
- `../../../configs/_base_/default_runtime.py`


In [ ]:
import os
import shutil
import subprocess
from pathlib import Path


def get_github_token():
    token = os.environ.get("GITHUB_TOKEN", "")
    if token:
        return token
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET_NAME) or ""
    except Exception:
        return ""


def copy_repo_fallback():
    if not REPO_INPUT_FALLBACK.exists():
        raise RuntimeError(
            "Không clone được GitHub và cũng không thấy REPO_INPUT_FALLBACK.\n"
            "Hãy bật Internet/make repo public, thêm Kaggle Secret GITHUB_TOKEN, "
            f"hoặc upload source tại {REPO_INPUT_FALLBACK}"
        )
    ignore = shutil.ignore_patterns(".git", "work_dirs", "__pycache__", "*.pyc", ".ipynb_checkpoints")
    shutil.copytree(REPO_INPUT_FALLBACK, REPO_DIR, ignore=ignore)
    print(f"[OK] Copied repo fallback: {REPO_INPUT_FALLBACK} -> {REPO_DIR}")


def clone_repo():
    if REPO_DIR.exists():
        print(f"[OK] Repo exists: {REPO_DIR}")
        return
    REPO_DIR.parent.mkdir(parents=True, exist_ok=True)

    public_cmd = ["git", "clone", REPO_GIT_URL, str(REPO_DIR)]
    print("$", " ".join(public_cmd))
    try:
        subprocess.check_call(public_cmd)
        print(f"[OK] Cloned public repo")
        return
    except subprocess.CalledProcessError as exc:
        print(f"[WARN] Public clone failed: {exc.returncode}")

    token = get_github_token()
    if token:
        private_cmd = ["git", "-c", f"http.extraHeader=Authorization: Bearer {token}", "clone", REPO_GIT_URL, str(REPO_DIR)]
        try:
            subprocess.check_call(private_cmd)
            print(f"[OK] Cloned private repo")
            return
        except subprocess.CalledProcessError as exc:
            print(f"[WARN] Private clone failed: {exc.returncode}")

    copy_repo_fallback()


def link_or_copy(src: Path, dst: Path):
    if dst.exists() or dst.is_symlink():
        print(f"[OK] Exists: {dst}")
        return
    if not src.exists():
        raise FileNotFoundError(f"Không thấy input: {src}")
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        dst.symlink_to(src, target_is_directory=src.is_dir())
        print(f"[OK] Symlink: {dst} -> {src}")
    except OSError:
        if src.is_dir():
            shutil.copytree(src, dst)
        else:
            shutil.copy2(src, dst)
        print(f"[OK] Copied: {src} -> {dst}")


def strip_bom_from_text_files():
    """Remove UTF-8 BOM that breaks Python 3.7 ast.parse on Kaggle."""
    patterns = ["configs/**/*.py", "projects/**/*.py", "tools/**/*.py", "tools/**/*.sh"]
    fixed = []
    for pattern in patterns:
        for path in REPO_DIR.glob(pattern):
            if not path.is_file():
                continue
            raw = path.read_bytes()
            if raw.startswith(b"\xef\xbb\xbf"):
                path.write_bytes(raw[3:])
                fixed.append(path)
    if fixed:
        print("[OK] Removed UTF-8 BOM from:")
        for path in fixed:
            print("  ", path.relative_to(REPO_DIR))
    else:
        print("[OK] No UTF-8 BOM found in config/source files")


def patch_configs_to_local_base():
    replacements = {
        "mmrotate::_base_/datasets/dior.py": "../../../configs/_base_/datasets/dior.py",
        "mmrotate::_base_/datasets/dota.py": "../../../configs/_base_/datasets/dota.py",
        "mmrotate::_base_/datasets/dotav15.py": "../../../configs/_base_/datasets/dotav15.py",
        "mmrotate::_base_/schedules/schedule_3x.py": "../../../configs/_base_/schedules/schedule_3x.py",
        "mmrotate::_base_/default_runtime.py": "../../../configs/_base_/default_runtime.py",
    }
    for name in JOBS:
        cfg = REPO_DIR / JOBS[name]["config"]
        text = cfg.read_text(encoding="utf-8")
        new_text = text
        for old, new in replacements.items():
            new_text = new_text.replace(old, new)
        if new_text != text:
            cfg.write_text(new_text, encoding="utf-8")
            print(f"[OK] Patched base paths: {cfg}")


def prepare_data_and_ckpts():
    if not DATA_ROOT.exists():
        raise FileNotFoundError(f"Không thấy DATA_ROOT: {DATA_ROOT}")
    CKPT_DIR.mkdir(parents=True, exist_ok=True)
    for name in selected_names():
        job = JOBS[name]
        link_or_copy(job["data_input"], REPO_DIR / job["data_link"])
        if CKPT_INPUT is None:
            print("[SKIP CKPT INPUT] Không dùng checkpoint ngoài; train từ đầu hoặc resume từ work_dirs.")
            continue
        ckpt_src = CKPT_INPUT / job["ckpt"]
        ckpt_dst = CKPT_DIR / job["ckpt"]
        if ckpt_src.exists():
            link_or_copy(ckpt_src, ckpt_dst)
        else:
            print(f"[WARN] Không thấy checkpoint gốc: {ckpt_src}; train từ đầu vẫn chạy được.")

clone_repo()
os.chdir(REPO_DIR)
strip_bom_from_text_files()
patch_configs_to_local_base()
prepare_data_and_ckpts()
print("cwd =", Path.cwd())


## 4. Setup môi trường

Không dùng shell script. Cell này cài trực tiếp các dependency cần thiết rồi `pip install -e .`.


In [ ]:
import importlib.util
import sys


def has_module(name):
    return importlib.util.find_spec(name) is not None


def run(cmd, cwd=REPO_DIR, env=None):
    cmd = [str(x) for x in cmd]
    print("$", " ".join(cmd))
    subprocess.check_call(cmd, cwd=cwd, env=env)


def env_ready():
    try:
        import torch, mmcv, mmdet, mmengine, mmrotate
        print("Environment ready:", torch.__version__, mmcv.__version__, mmdet.__version__, mmrotate.__version__)
        return True
    except Exception as exc:
        print("Environment not ready:", repr(exc))
        return False

if RUN_SETUP:
    if not env_ready():
        run([sys.executable, "-m", "pip", "install", "-U", "pip", "setuptools", "wheel"])
        req_build = REPO_DIR / "requirements/build.txt"
        if req_build.exists():
            run([sys.executable, "-m", "pip", "install", "-r", str(req_build)])
        run([sys.executable, "-m", "pip", "install", "packaging", "matplotlib", "pycocotools", "six", "terminaltables", "scipy", "scikit-learn", "imagecorruptions"])
        if not has_module("openmim"):
            run([sys.executable, "-m", "pip", "install", "-U", "openmim"])
        if not has_module("mmcv"):
            run([sys.executable, "-m", "mim", "install", "mmcv>=2.0.0rc2,<2.1.0"])
        if not has_module("mmengine"):
            run([sys.executable, "-m", "pip", "install", "mmengine>=0.1.0"])
        if not has_module("mmdet"):
            run([sys.executable, "-m", "pip", "install", "mmdet>=3.0.0rc5,<3.1.0"])
        run([sys.executable, "-m", "pip", "install", "-v", "-e", "."])
    env_ready()
else:
    print("[SKIP SETUP]")


## 5. Hàm chạy train/test/analysis trực tiếp


In [ ]:
def latest_ckpt(path: Path):
    if not path.exists():
        return None
    latest = path / "latest.pth"
    if latest.exists():
        return latest
    ckpts = sorted(path.glob("epoch_*.pth"), key=lambda p: int(p.stem.split("_")[-1]) if p.stem.split("_")[-1].isdigit() else -1)
    return ckpts[-1] if ckpts else None


def count_files(path: Path):
    return sum(1 for p in path.rglob("*") if p.is_file()) if path.exists() else 0


def cfg_options():
    opts = [
        f"train_dataloader.batch_size={BATCH_SIZE}",
        f"train_dataloader.num_workers={NUM_WORKERS}",
        "val_dataloader.batch_size=1",
        f"val_dataloader.num_workers={NUM_WORKERS}",
        "test_dataloader.batch_size=1",
        f"test_dataloader.num_workers={NUM_WORKERS}",
    ]
    if MAX_EPOCHS is not None:
        opts.append(f"train_cfg.max_epochs={MAX_EPOCHS}")
    if LR is not None:
        opts.append(f"optim_wrapper.optimizer.lr={LR}")
    return opts


def run_train_job(name, job, root):
    train_dir = root / "train"
    done = root / ".train.done"
    if not RUN_TRAIN:
        print("[SKIP TRAIN]", name)
        return latest_ckpt(train_dir)
    ckpt = latest_ckpt(train_dir)
    if done.exists() and ckpt and not FORCE:
        print("[SKIP TRAIN DONE]", name, ckpt)
        return ckpt
    if ckpt and not FORCE:
        print("[SKIP TRAIN CKPT]", name, ckpt)
        done.touch()
        return ckpt
    train_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "tools/train.py", job["config"], "--work-dir", str(train_dir), "--cfg-options", *cfg_options()]
    print("[RUN TRAIN]", job["title"])
    run(cmd)
    done.touch()
    return latest_ckpt(train_dir)


def resolve_ckpt(name, job, root):
    trained = latest_ckpt(root / "train")
    if trained:
        return trained
    if CKPT_INPUT is None:
        return None
    external = CKPT_DIR / job["ckpt"]
    return external if external.exists() else None

def run_test_job(name, job, root, ckpt):
    test_dir = root / "test"
    pred = test_dir / "predictions.pkl"
    done = root / ".test.done"
    if not RUN_TEST:
        print("[SKIP TEST]", name)
        return pred
    if done.exists() and pred.exists() and not FORCE:
        print("[SKIP TEST DONE]", name)
        return pred
    if not ckpt or not Path(ckpt).exists():
        print("[SKIP TEST NO CKPT]", name)
        return pred
    test_dir.mkdir(parents=True, exist_ok=True)
    cmd = [sys.executable, "tools/test.py", job["config"], str(ckpt), "--work-dir", str(test_dir), "--out", str(pred), "--cfg-options", "test_dataloader.batch_size=1", f"test_dataloader.num_workers={NUM_WORKERS}"]
    print("[RUN TEST]", job["title"])
    run(cmd)
    done.touch()
    return pred


def run_optional_analysis(name, job, root, ckpt, pred):
    chart_dir = root / "charts"
    bbox_dir = root / "images" / "bboxes"
    heatmap_dir = root / "images" / "heatmaps"
    image_source = REPO_DIR / job["image_source"]

    if RUN_CHARTS and not (root / ".charts.done").exists():
        plot_script = REPO_DIR / "tools/analysis_tools/plot_rqformer_charts.py"
        logs = list((root / "train").rglob("*.json")) + list((root / "test").rglob("*.json"))
        if plot_script.exists() and logs:
            chart_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(plot_script), *map(str, logs), "--out-dir", str(chart_dir), "--title", job["title"]])
            (root / ".charts.done").touch()
        else:
            print("[SKIP CHARTS] missing script or json logs")

    if RUN_CONFUSION and not (root / ".confusion.done").exists():
        cm_script = REPO_DIR / "tools/analysis_tools/confusion_matrix.py"
        if cm_script.exists() and pred.exists():
            out_dir = chart_dir / "confusion_matrix"
            out_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(cm_script), job["config"], str(pred), str(out_dir), "--score-thr", str(SCORE_THR), "--cfg-options", "test_dataloader.batch_size=1", f"test_dataloader.num_workers={NUM_WORKERS}"])
            (root / ".confusion.done").touch()
        else:
            print("[SKIP CONFUSION] missing script or predictions")

    if RUN_BBOX and not (root / ".bbox.done").exists():
        bbox_script = REPO_DIR / "tools/analysis_tools/visualize_rqformer_bboxes.py"
        if bbox_script.exists() and ckpt and Path(ckpt).exists() and image_source.exists():
            bbox_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(bbox_script), job["config"], str(ckpt), str(image_source), "--out-dir", str(bbox_dir), "--device", "cuda:0", "--score-thr", str(SCORE_THR), "--max-images", str(SAMPLE_IMAGES)])
            (root / ".bbox.done").touch()
        else:
            print("[SKIP BBOX] missing script, ckpt, or image source")

    if RUN_HEATMAP and not (root / ".heatmap.done").exists():
        heat_script = REPO_DIR / "tools/analysis_tools/visualize_rroi_attention.py"
        if heat_script.exists() and ckpt and Path(ckpt).exists() and image_source.exists():
            heatmap_dir.mkdir(parents=True, exist_ok=True)
            run([sys.executable, str(heat_script), job["config"], str(ckpt), str(image_source), "--out-dir", str(heatmap_dir), "--device", "cuda:0", "--score-thr", str(SCORE_THR), "--max-images", str(SAMPLE_IMAGES)])
            (root / ".heatmap.done").touch()
        else:
            print("[SKIP HEATMAP] missing script, ckpt, or image source")

    print("Charts:", count_files(chart_dir), "BBoxes:", count_files(bbox_dir), "Heatmaps:", count_files(heatmap_dir))


## 7. Show toàn bộ kết quả và visualization trong notebook

Cell này không chạy lại train/test. Nó chỉ đọc những gì đã sinh trong `work_dirs` rồi:

- In checkpoint, prediction, log, chart, bbox, heatmap ra terminal.
- Hiển thị toàn bộ ảnh chart/bbox/heatmap đã sinh trong notebook.
- Giữ nguyên file trong `work_dirs` để cell zip phía dưới đóng gói download.


In [ ]:
from pathlib import Path

try:
    from IPython.display import display, Image as IPyImage, Markdown
except Exception:
    display = None
    IPyImage = None
    Markdown = None

IMAGE_EXTS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}
TEXT_EXTS = {".log", ".txt", ".md", ".json"}


def list_files(path: Path, exts=None):
    if not path.exists():
        return []
    files = [p for p in path.rglob("*") if p.is_file()]
    if exts is not None:
        files = [p for p in files if p.suffix.lower() in exts]
    return sorted(files)


def print_section(title):
    print("\n" + "=" * 80)
    print(title)
    print("=" * 80)


def tail_text(path: Path, n=30):
    try:
        lines = path.read_text(encoding="utf-8", errors="replace").splitlines()
        return "\n".join(lines[-n:])
    except Exception as exc:
        return f"[cannot read {path}: {exc}]"


def display_images(title, files):
    print_section(title)
    if not files:
        print("[NONE]")
        return
    for path in files:
        print(path)
        if display is not None and IPyImage is not None:
            try:
                display(Markdown(f"**{path.relative_to(WORK_ROOT)}**"))
                display(IPyImage(filename=str(path)))
            except Exception as exc:
                print(f"[DISPLAY SKIP] {path}: {exc}")


def show_job_outputs(name):
    root = WORK_ROOT / "rroiformer" / name
    print_section(f"JOB OUTPUT: {name}")
    print("Root:", root)
    if not root.exists():
        print("[MISSING ROOT]")
        return

    checkpoints = list_files(root / "train", {".pth"})
    predictions = list_files(root / "test", {".pkl"})
    logs = list_files(root, {".log", ".txt", ".json"})
    charts = list_files(root / "charts", IMAGE_EXTS)
    bboxes = list_files(root / "images" / "bboxes", IMAGE_EXTS)
    heatmaps = list_files(root / "images" / "heatmaps", IMAGE_EXTS)

    print("Checkpoints:")
    for f in checkpoints:
        print(" ", f)
    print("Predictions:")
    for f in predictions:
        print(" ", f)
    print("Logs/Text/JSON:")
    for f in logs:
        print(" ", f)
    print(f"Charts: {len(charts)} | BBoxes: {len(bboxes)} | Heatmaps: {len(heatmaps)}")

    # Print tail of the newest readable log/text file so metrics are visible in terminal.
    readable_logs = [p for p in logs if p.suffix.lower() in {".log", ".txt", ".md"}]
    if readable_logs:
        newest = max(readable_logs, key=lambda p: p.stat().st_mtime)
        print_section(f"TAIL LOG: {newest}")
        print(tail_text(newest, n=40))

    display_images(f"CHARTS: {name}", charts)
    display_images(f"BBOX VISUALIZATION: {name}", bboxes)
    display_images(f"HEATMAP VISUALIZATION: {name}", heatmaps)


def show_all_workflow_outputs():
    for name in selected_names():
        show_job_outputs(name)

    summary = WORK_ROOT / "rroiformer" / "summary_results.md"
    if summary.exists():
        print_section("SUMMARY RESULTS")
        text = summary.read_text(encoding="utf-8", errors="replace")
        print(text)
        if display is not None and Markdown is not None:
            display(Markdown(text))

show_all_workflow_outputs()


## 6. Chạy toàn bộ workflow


In [ ]:
os.environ["CUDA_VISIBLE_DEVICES"] = str(DEVICE)
WORK_ROOT.mkdir(parents=True, exist_ok=True)

for name in selected_names():
    job = JOBS[name]
    root = WORK_ROOT / "rroiformer" / name
    (root / "logs").mkdir(parents=True, exist_ok=True)
    print("=" * 80)
    print("[JOB]", job["title"])
    print("Output:", root)

    cfg = REPO_DIR / job["config"]
    data = REPO_DIR / job["data_link"]
    if not cfg.exists():
        raise FileNotFoundError(cfg)
    if not data.exists():
        raise FileNotFoundError(data)

    trained = run_train_job(name, job, root)
    ckpt = resolve_ckpt(name, job, root)
    print("[USING CKPT]", ckpt)
    pred = run_test_job(name, job, root, ckpt)
    run_optional_analysis(name, job, root, ckpt, pred)


## 8. Summary results


In [ ]:
summary_script = REPO_DIR / "tools/analysis_tools/write_rqformer_summary.py"
summary = WORK_ROOT / "rroiformer" / "summary_results.md"
if RUN_SUMMARY and summary_script.exists():
    run([sys.executable, str(summary_script), "--repo-dir", str(REPO_DIR), "--work-root", str(WORK_ROOT), "--out", str(summary)])
    print(summary.read_text(encoding="utf-8", errors="replace"))
else:
    print("[SKIP SUMMARY] missing script or RUN_SUMMARY=0")


## 9. Xem cây output và zip kết quả


In [ ]:
def tree(path: Path, max_depth=4):
    if not path.exists():
        print("Not found:", path)
        return
    base_depth = len(path.parts)
    for p in sorted(path.rglob("*")):
        depth = len(p.parts) - base_depth
        if depth > max_depth:
            continue
        indent = "  " * max(depth - 1, 0)
        print(f"{indent}{p.name}{'/' if p.is_dir() else ''}")

print("Output root:", WORK_ROOT)
tree(WORK_ROOT / "rroiformer", max_depth=4)

archive_base = Path("/kaggle/working/rqformer2_full_workflow_outputs")
zip_path = shutil.make_archive(str(archive_base), "zip", root_dir=str(WORK_ROOT))
print("Saved zip:", zip_path)


## 10. Chạy nhanh

Chỉ train DOTA-v1.0:

```python
DATASETS = "dotav1_0"
RUN_TEST = 0
RUN_CHARTS = 0
RUN_CONFUSION = 0
RUN_BBOX = 0
RUN_HEATMAP = 0
RUN_SUMMARY = 0
```

Chỉ test/visualize từ checkpoint đã có:

```python
DATASETS = "dotav1_0"
RUN_TRAIN = 0
RUN_TEST = 1
RUN_BBOX = 1
RUN_HEATMAP = 1
```
